# Speculative Decoding

> **Status:** Content notebook — hands-on benchmarks require a GPU with at least 24 GB VRAM.

## Learning Objectives

By the end of this notebook, you will be able to:

- [ ] Explain speculative decoding and why it reduces latency without changing output quality
- [ ] Describe the role of the draft model and the verification step
- [ ] Implement a basic speculative decoding loop from scratch
- [ ] Enable speculative decoding in vLLM and measure the speedup
- [ ] Understand when speculative decoding helps and when it doesn't
- [ ] Describe alternative approaches: Medusa heads, EAGLE, and prompt lookup decoding

---

## Prerequisites

- [03_kv_cache_paged_attention.ipynb](../03_kv_cache_paged_attention/03_kv_cache_paged_attention.ipynb)
- Familiarity with autoregressive decoding

---

## 1. The Latency Problem in Autoregressive Decoding

Standard decoding generates **one token per forward pass** of the full model.
The forward pass is often memory-bandwidth bound (especially with KV cache), not
compute bound — meaning the large model is underutilized on each single-token step.

Speculative decoding exploits this by using a **fast draft model** to propose
multiple tokens at once, then **verifying** them in a single parallel pass of the
large target model.

---

## 2. The Algorithm

```
Algorithm: Speculative Decoding (Chen et al., 2023)
--------------------------------------------------
Given: target model M_target, draft model M_draft, speculation length γ

For each decoding step:
  1. Draft: Run M_draft autoregressively for γ steps → tokens [t1, t2, ..., tγ]
  2. Verify: Run M_target on all γ draft tokens in PARALLEL (one forward pass)
  3. Accept/Reject: For each draft token ti:
       - Sample qi from M_target(ti | context)
       - Sample pi from M_draft(ti | context)
       - Accept with probability min(1, qi/pi)
       - On first rejection: resample from adjusted distribution and stop
  4. Result: Between 1 and γ+1 tokens accepted per target model forward pass
```

Key property: **The output distribution is identical to standard sampling from M_target.**
No quality loss — only latency improvement.

---

## 3. Speedup Analysis

Speedup depends on the **acceptance rate** — how often draft tokens match the target.

```python
# Theoretical speedup formula
# alpha = mean acceptance rate per draft token
# gamma = number of draft tokens

def expected_speedup(alpha: float, gamma: int) -> float:
    # Expected tokens per target model call
    expected_tokens = (1 - alpha**(gamma + 1)) / (1 - alpha)
    # Cost: 1 target call + gamma draft calls (draft is ~10-100x cheaper)
    # Speedup vs baseline (1 token per target call)
    return expected_tokens  # simplified (ignoring draft cost)

# Example: alpha=0.8, gamma=4
print(f"Speedup ≈ {expected_speedup(0.8, 4):.2f}x")  # ~3.36x
```

Typical real-world speedup: **1.5–3.5x** depending on task, draft model quality,
and hardware.

---

## 4. Draft Model Options

| Draft strategy | Description | Notes |
|----------------|-------------|-------|
| Small model (same family) | e.g., Llama-3.2-1B drafts for Llama-3-70B | Most common; best acceptance rates for matched families |
| N-gram / prompt lookup | Match draft tokens against prompt text | Works well for long-document summarization |
| Medusa heads | Add parallel prediction heads to the target | No separate model needed; faster at inference |
| EAGLE | Lightweight feature predictor on target hidden states | High acceptance rate with minimal overhead |

---

## 5. Speculative Decoding in vLLM

```python
# vLLM >= 0.4.0 supports speculative decoding
from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Meta-Llama-3-70B-Instruct",
    speculative_model="meta-llama/Llama-3.2-1B-Instruct",
    num_speculative_tokens=5,
    dtype="bfloat16",
)

outputs = llm.generate(
    ["Explain the attention mechanism in transformers."],
    SamplingParams(temperature=0.0, max_tokens=256),
)
print(outputs[0].outputs[0].text)
```

```python
# Benchmark with and without speculative decoding
import time

# Without speculative decoding
llm_base = LLM(model="meta-llama/Meta-Llama-3-70B-Instruct")
start = time.time()
_ = llm_base.generate(prompts, SamplingParams(max_tokens=200))
baseline_time = time.time() - start

# With speculative decoding
llm_spec = LLM(
    model="meta-llama/Meta-Llama-3-70B-Instruct",
    speculative_model="meta-llama/Llama-3.2-1B-Instruct",
    num_speculative_tokens=5,
)
start = time.time()
_ = llm_spec.generate(prompts, SamplingParams(max_tokens=200))
spec_time = time.time() - start

print(f"Baseline: {baseline_time:.1f}s | Speculative: {spec_time:.1f}s | Speedup: {baseline_time/spec_time:.2f}x")
```

---

## 6. When Does It Help?

**Speculative decoding helps most when:**
- The task has predictable outputs (code generation, summarization, translation)
- The draft model is from the same family as the target
- Batch size is small (single-user or low-concurrency serving)

**Speculative decoding helps least when:**
- Creative or highly varied outputs (high temperature, story generation)
- Large batch sizes where the target model is already compute-saturated
- The draft model's token distribution diverges significantly from the target

---

## Exercises

1. Implement a simple speculative decoding loop with HuggingFace transformers using
   a Llama-3.2-1B draft and Llama-3-8B target.
2. Measure acceptance rate for different tasks: code generation vs creative writing.
3. Compare Medusa (if available) against the draft-model approach on the same benchmark.

---

## References

- [Speculative Decoding paper (Chen et al., 2023)](https://arxiv.org/abs/2302.01318)
- [Medusa paper](https://arxiv.org/abs/2401.10774)
- [EAGLE paper](https://arxiv.org/abs/2401.15077)
- [vLLM speculative decoding docs](https://docs.vllm.ai/en/latest/features/spec_decode.html)

## What Comes Next

- [06_serving_runtimes_comparison.ipynb](../06_serving_runtimes_comparison/06_serving_runtimes_comparison.ipynb) — vLLM vs TRT-LLM vs SGLang trade-offs
- [07_prefix_caching_chunked_prefill.ipynb](../07_prefix_caching_chunked_prefill/07_prefix_caching_chunked_prefill.ipynb) — Prefix caching and chunked prefill
